# VocalCoach Colab Training

Multi-task conformer training: pitch + VAD + technique (+ quality, note heads in later stages).

**Training strategy:** Differential LR throughout — backbone gets a low LR to protect pitch/VAD
representations while technique/quality heads train faster.

**Checkpoint strategy:** Checkpoints write to local disk (`/content/runs/`) for fast I/O.
A Drive-copy cell runs after each stage. If the session disconnects, re-run cells 1–4 and
`--resume` picks up from the last Drive checkpoint automatically.

**Run cells in order.** Cells 1–4 are setup (re-run at the start of every new session).

---

### Completed runs (results in VOCALCOACH_RESULTS.md)

| # | Run | mF1 (VS) | mF1 (GT) | VDR | VAD Acc | Notes |
|---|---|---|---|---|---|---|
| 23 | `stage1_conformer_96` | 0.397 | — | 88.3% | 81.7% | Stage 1 h=96 |
| 24 | `stage1_conformer_128` | 0.412 | — | 89.3% | 81.9% | ❌ VAD head dead (max=0.005) |
| 25 | `stage2_w1_joint_difflr` | 0.769 | 0.190 | 79.0% | 82.4% | **W2_BASE** — best technique |
| 26 | `stage2_w1_difflr_v2` | 0.747 | 0.190 | 78.0% | 81.7% | continued difflr |
| 27 | `stage2_w1_supcon` | 0.745 | 0.535 | 68.4% | 82.1% | SupCon — VDR crash −10.6pp |

### New runs (W1_NEW block — run before W2A)

| Run | Expected | Notes |
|---|---|---|
| `stage1_conformer_128_vadfix` | VDR>89.3% AND VAD Acc>81.9% | Stage 1 w/ w-vad=0.2 + VocalSet pitch diversity |
| `stage2_w2_new_joint` | mF1≥0.769, VDR≥79%, VAD Acc≥82.4% | Stage 2 off new backbone, balanced |
| `stage2_w2_new_warmup` | VDR recovery to 85%+ | Only if W2_NEW VDR < 79% |

**Backbone acceptance rule:** only promote `stage1_conformer_128_vadfix` to W2_NEW_BASE
if **both** VDR > 89.3% AND VAD Acc > 81.9% improve together. One metric up / one down = retrain.

---

### Data upload (one-time, from local machine)

```bash
cd ~/NanoPitch-MusicalAI
zip -1 NanoPitch_data.zip \
    data/clean.npz \
    data/noise.npz \
    data/test.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz \
    data/annotated_vocalset/technique_train.npz \
    data/annotated_vocalset/technique_test.npz \
    data/gtsinger_technique/technique_gtsinger_train.npz \
    data/gtsinger_technique/technique_gtsinger_test.npz \
    data/gtsinger_technique/technique_train.npz
```

Upload `NanoPitch_data.zip` to `My Drive/musicalAI/vocalCoach/`.

Expected Drive layout:
```
My Drive/musicalAI/vocalCoach/
  NanoPitch_data.zip
  NanoPitch-runs/
    stage1_conformer_128/           ← old W1_BASE (dead VAD, do not use as backbone)
    stage1_conformer_128_vadfix/    ← W1_NEW_BASE (VAD-fixed, pending)
    stage2_w1_joint_difflr/         ← W2_BASE (#25, mF1=0.769)
    stage2_w2_new_joint/            ← W2_NEW (off new backbone)
    stage2_w2_new_warmup/           ← W2_NEW_B (VDR recovery fallback)
    stage2_w2a_balanced/            ← W2A (in progress, off old W2_BASE)
    stage2_w2b_warmup/              ← W2B (pending)
    stage2_w2c_curriculum/          ← W2C (pending)
    stage2_w2d_supcon_light/        ← W2D (pending)
    stage2_w1_quality_v2/           ← quality head (pending)
    stage2_v4_note/                 ← note head (pending)
```

## Cell 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 64 if vram_gb > 30 else 32 if vram_gb > 15 else 16
NUM_WORKERS = 8
print(f"\nRecommended batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'
os.makedirs(RUNS_DIR, exist_ok=True)

# Files in the updated NanoPitch_data.zip (uploaded to Drive)
files_needed = {
    'data/clean.npz':                                           '/content/data/clean.npz',
    'data/noise.npz':                                           '/content/data/noise.npz',
    'data/test.npz':                                            '/content/data/test.npz',
    'data/vocalset/technique_train.npz':                        '/content/data/vocalset/technique_train.npz',
    'data/vocalset/technique_test.npz':                         '/content/data/vocalset/technique_test.npz',
    'data/annotated_vocalset/technique_train.npz':              '/content/data/annotated_vocalset/technique_train.npz',
    'data/annotated_vocalset/technique_test.npz':               '/content/data/annotated_vocalset/technique_test.npz',
    'data/gtsinger_technique/technique_gtsinger_train.npz':     '/content/data/gtsinger_technique/technique_gtsinger_train.npz',
    'data/gtsinger_technique/technique_gtsinger_test.npz':      '/content/data/gtsinger_technique/technique_gtsinger_test.npz',
    'data/gtsinger_technique/technique_train.npz':              '/content/data/gtsinger_technique/technique_train.npz',
}

missing = [name for name, dest in files_needed.items() if not os.path.exists(dest)]

if missing:
    print(f"Extracting {len(missing)} missing file(s) from NanoPitch_data.zip ...")
    with zipfile.ZipFile(f'{DRIVE_ROOT}/NanoPitch_data.zip', 'r') as z:
        for name in missing:
            os.makedirs(os.path.dirname(files_needed[name]), exist_ok=True)
            z.extract(name, '/content/')
            print(f"  ✓ {name}")
else:
    print("All data files already present — skipping extraction.")

!ls -lh /content/data/
!ls -lh /content/data/vocalset/
!ls -lh /content/data/annotated_vocalset/
!ls -lh /content/data/gtsinger_technique/

## Cell 3 — Clone or update repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/NanoPitch-MusicalAI'
REPO_URL = 'https://github.com/rajat17-personal/NanoPitch-MusicalAI'
BRANCH   = 'feat/finalProject'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet
print("Setup complete.")

## Cell 4 — Verify data loads correctly

In [ ]:
import numpy as np

def check(label, path, key='lengths'):
    try:
        d = np.load(path)
        n = d[key].shape[0]
        print(f"  ✓  {label:<48} {n:>6} clips  keys={list(d.keys())}")
    except Exception as e:
        print(f"  ✗  {label:<48} MISSING or ERROR: {e}")

print("=== Data verification ===")
check('clean.npz',                              '/content/data/clean.npz')
check('noise.npz',                              '/content/data/noise.npz')
check('test.npz',                               '/content/data/test.npz', key='clips')
check('vocalset/technique_train.npz',           '/content/data/vocalset/technique_train.npz')
check('vocalset/technique_test.npz',            '/content/data/vocalset/technique_test.npz')
check('annotated_vocalset/technique_train.npz', '/content/data/annotated_vocalset/technique_train.npz')
check('annotated_vocalset/technique_test.npz',  '/content/data/annotated_vocalset/technique_test.npz')
check('gtsinger_technique/technique_train.npz', '/content/data/gtsinger_technique/technique_train.npz')
check('gtsinger_technique/gtsinger_train.npz',  '/content/data/gtsinger_technique/technique_gtsinger_train.npz')
check('gtsinger_technique/gtsinger_test.npz',   '/content/data/gtsinger_technique/technique_gtsinger_test.npz')

---
## Helper: save run to Drive

Reusable function — call after any training cell completes.

```python
save_to_drive("run_name")
```

In [ ]:
import shutil, os

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'

def save_to_drive(run_name):
    src = f'/content/runs/{run_name}'
    dst = f'{RUNS_DIR}/{run_name}'
    os.makedirs(RUNS_DIR, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Saved {run_name} → Drive")

def restore_from_drive(run_name):
    src = f'{RUNS_DIR}/{run_name}'
    dst = f'/content/runs/{run_name}'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Restored {run_name} ← Drive")

print("save_to_drive / restore_from_drive ready.")

---
## W1 — Stage 2D: quality scoring head (Variant 2 — multi-dim ccmusic + MSE distil)

**Is it implemented?** Yes — fully wired in `train.py`. The quality head uses `--probe-mode` which freezes the backbone + pitch/VAD/technique heads and trains only `head_quality`. This means it **branches off the best technique checkpoint independently** — it does not compete with technique training.

**Which checkpoint to start from?** Use the **best technique run** — `stage2_w1_joint_difflr` (#25, mF1=0.769). Do NOT use the SupCon run (#27) as the starting point; the VDR crash means its backbone representations are less clean.

**Variant 2** (`--quality-variant 2`) trains a 9-dim head matching CCMusicNet expert dimensions:
- Stage 1 (epochs 1–30): MSE on SingMOS-Pro AudioScore pseudo-labels (`quality_mse.npz`)
- Stage 2 (epochs 31+): CCMusicNet 9-dim expert labels (`quality_ccmusic.npz`) + contrastive ranking on PopBuTFy pairs (`quality_pairs.npz`)

**Data requirements** — run `scripts/prepareQualityData.py` locally first to generate these NPZs, then upload to Drive:
```bash
# Local (one-time):
python scripts/prepareQualityData.py \
    --singmos-scores-json  data/singmos_scores.json \
    --ccmusic-wavs-dir     data/ccmusic \
    --popbutfy-dir         data/popbutfy \
    --baselines-json       data/combined_eval_baselines.json \
    --output-dir           data/quality

# Then add to zip:
zip -1 NanoPitch_data.zip \
    data/quality/quality_mse.npz \
    data/quality/quality_ccmusic.npz \
    data/quality/quality_pairs.npz
```

In [ ]:
import os, shutil

# Copy quality NPZs from Drive to local /content/data/quality/
QUALITY_FILES = ['quality_mse.npz', 'quality_ccmusic.npz', 'quality_pairs.npz']
os.makedirs('/content/data/quality', exist_ok=True)
for f in QUALITY_FILES:
    src = f'{DRIVE_ROOT}/NanoPitch-runs/quality/{f}'   # adjust path if you uploaded elsewhere
    dst = f'/content/data/quality/{f}'
    if not os.path.exists(dst):
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"Copied {f}")
        else:
            print(f"MISSING: {src}  — run scripts/prepareQualityData.py locally first")
    else:
        print(f"Already present: {f}")

# The quality head is trained in probe mode:
#   - backbone, pitch/VAD/technique heads are ALL FROZEN
#   - only head_quality trains
# Start from the best technique checkpoint (#25) — clean backbone representations.
# --quality-variant 2: 9-dim CCMusicNet head with MSE pretraining + contrastive ranking
# --quality-epochs-mse 30: epochs 1-30 = MSE on AudioScore scalars, then CCMusicNet + ranking
QUALITY_BASE = "stage2_w1_joint_difflr"   # #25 — best technique F1, cleanest backbone
restore_from_drive(QUALITY_BASE)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --probe-mode \
    --quality-variant 2 \
    --quality-mse-npz     /content/data/quality/quality_mse.npz \
    --quality-ccmusic-npz /content/data/quality/quality_ccmusic.npz \
    --quality-pairs-npz   /content/data/quality/quality_pairs.npz \
    --quality-epochs-mse 30 \
    --w-quality-mse 1.0 --w-ranking 1.0 --ranking-margin 0.5 \
    --output-dir /content/runs/stage2_w1_quality_v2 \
    --epochs 80 --batch-size 64 --num-workers 8 --seq-len 600 \
    --lr 3e-4 \
    --pitch-sigma 0.8 --augment none \
    --patience 20 \
    --resume /content/runs/{QUALITY_BASE}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w1_quality_v2")

---
## W1_NEW — Stage 1: wider backbone with VAD-aware training (new baseline)

### Why this replaces stage1_conformer_128 as W1_BASE

`stage1_conformer_128` (#24) has the best pitch scores in the table (VDR=89.3%, RPA=99.4%) but its
VAD head outputs near-zero everywhere (max=0.005). The wider model (hidden=128) fits the pitch task
so fast on GTSinger-only data that the VAD head receives almost no gradient — the pitch loss
(w-pitch=2) dominates the VAD loss (w-vad=0.05) by a factor of 40×.

**Two-fix approach:**

1. **Raise `--w-vad` from 0.05 → 0.2** — forces the VAD head to compete with pitch gradients
   throughout training instead of being swamped. Only needed for hidden=128; narrower models
   converge more slowly and the default weight is sufficient.

2. **Add VocalSet `clean.npz` via `--technique-dirs` with `--w-technique 0`** — adds 824 all-voiced
   clips of diverse singing to Stage 1, giving the pitch head more varied voiced-frame examples
   without introducing technique supervision. VocalSet has no silence frames, so this does not
   bias the voiced/silence balance for VAD — it only widens the pitch distribution the backbone
   sees before Stage 2.

**Acceptance criterion:** only promote this checkpoint to W2_NEW_BASE if **both** VDR and VAD Acc
improve jointly vs stage1_conformer_128 (VDR=89.3%, VAD Acc=81.9%). If only one improves, the
backbone is still trading tasks against each other.

**Data upload (one-time, if not already in zip):**
```bash
# VocalSet clean.npz — add to NanoPitch_data.zip if missing:
zip -1 NanoPitch_data.zip data/vocalset/technique_train.npz   # already included
# No new upload needed — we use technique_train.npz with w-technique=0
```

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --w-technique 0 \
    --output-dir /content/runs/stage1_conformer_128_vadfix \
    --epochs 150 --batch-size 64 --num-workers 8 --seq-len 600 \
    --lr 1e-3 \
    --w-vad 0.2 --w-pitch 2 \
    --vad-pos-weight 2.3 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30

# After training completes, check these two numbers before promoting to W2_NEW_BASE:
#   VAD Acc (clean) — must be > 81.9% (stage1_conformer_128 baseline)
#   VDR     (clean) — must be > 89.3% (stage1_conformer_128 baseline)
# If both improve: set W1_NEW_BASE = "stage1_conformer_128_vadfix" in the W2_NEW cell below.
# If only one improves: do not use as backbone — retrain with adjusted w-vad.

In [ ]:
save_to_drive("stage1_conformer_128_vadfix")

---
## W2_NEW — Stage 2: technique joint training off new backbone

Only run this cell after `stage1_conformer_128_vadfix` passes the dual acceptance criterion
(both VDR > 89.3% AND VAD Acc > 81.9%). Update `W1_NEW_BASE` below if the run name changes.

Same recipe as `stage2_w1_joint_difflr` (#25, best technique run) but starting from the
VAD-fixed backbone. Uses `--balance-datasets` from the start since that was the main
overfitting fix identified in the W2 analysis — no separate W2A step needed here.

**Expected vs #25 baseline:** mF1 ≥ 0.769, VDR ≥ 79.0%, VAD Acc ≥ 82.4%.
If VDR drops below 79% the backbone representations were disrupted by technique gradients
and you should try W2_NEW_B (warmup variant) instead.

In [ ]:
W1_NEW_BASE = "stage1_conformer_128_vadfix"   # update if run name changes
restore_from_drive(W1_NEW_BASE)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --output-dir /content/runs/stage2_w2_new_joint \
    --epochs 80 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 6e-4 --lr-backbone 1e-4 \
    --w-vad 0.2 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 25 \
    --resume /content/runs/{W1_NEW_BASE}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w2_new_joint")

---
## W2_NEW_B — Stage 2 warmup variant (if W2_NEW VDR < 79%)

Only run if `stage2_w2_new_joint` shows VDR regression below 79.0%. Adds `--warmup-heads-epochs 40`
so pitch/VAD reconverge for 40 epochs before technique gradients compete. Increases total epochs
to 120 to compensate for the delayed technique start. Everything else is identical to W2_NEW.

In [ ]:
# Only run if W2_NEW VDR < 79% — otherwise skip to W2A
restore_from_drive(W1_NEW_BASE)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --warmup-heads-epochs 40 \
    --output-dir /content/runs/stage2_w2_new_warmup \
    --epochs 120 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 6e-4 --lr-backbone 1e-4 \
    --w-vad 0.2 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 30 \
    --resume /content/runs/{W1_NEW_BASE}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w2_new_warmup")

---
## W2 — Analysis of W1 results and next runs

### W1 findings (runs #25, #26, #27)

| Run | mF1 (VocalSet) | mF1 (GTSinger) | VDR | RPA |
|---|---|---|---|---|
| #25 `w1_joint_difflr` | **0.769** | 0.190 | 79.0% | 98.6% |
| #26 `w1_difflr_v2` | 0.747 | 0.190 | 78.0% | 98.8% |
| #27 `w1_supcon` | 0.745 | **0.535** | 68.4% | 98.4% |

**Key conclusions:**
- **#25 is W2_BASE**: best VocalSet F1 (0.769), acceptable VDR (79%). Use this as the starting checkpoint.
- **GTSinger mF1=0.190 is overfitting**: VocalSet dominates technique training (824 clips vs 9601 GTSinger). `--balance-datasets` is the primary fix.
- **#27 SupCon VDR crash (−10.6 pp)**: SupCon + GTSinger-tech combination pulls backbone into technique-space. The falsetto=0.829 in GTSinger is a pitch-range artifact, not learned technique. SupCon needs lower weight or a frozen-backbone mode.
- **No OOD eval set**: GTSinger held-out is the best proxy. Track the ratio VocalSet_F1/GTSinger_F1 — currently 4:1, target 2:1.

### W2 strategy

- **W2A** (run first): `--balance-datasets` + all technique sources off #25. Expected GTSinger F1 → 0.35–0.45.
- **W2B**: warmup heads 40 epochs from clean stage1_conformer_128 backbone to recover VDR.
- **W2C**: GTSinger-only pretrain curriculum (`--pretrain-data-dir`), then fine-tune on combined.
- **V4**: note segmentation head off best W2 checkpoint (needs `note_train.npz` with onset/offset labels).

In [ ]:
# W2_BASE = run #25 (best technique checkpoint, VDR=79.0%, mF1=0.769)
# W1_BASE = stage1_conformer_128 (clean pitch backbone, VDR=89.3%) — needed for W2B/W2C
W2_BASE = "stage2_w1_joint_difflr"
W1_BASE = "stage1_conformer_128"

restore_from_drive(W2_BASE)
print(f"Ready: /content/runs/{W2_BASE}/checkpoints/")

---
## W2A — Balanced datasets (overfitting fix, Tier 1)

**Problem**: VocalSet technique (824 clips) dominates training over GTSinger-tech (9601 clips).
`--balance-datasets` uses WeightedRandomSampler so each source contributes equally per batch.

**Expected**: GTSinger held-out mF1 0.190 → 0.35–0.45, VocalSet mF1 may drop ~0.03–0.05 but model generalises better. VDR should stay ~79% since we resume from #25 with the same LRs.

New: `--technique-dirs` now includes `gtsinger_technique` (GTSinger technique labels) in addition to VocalSet. This gives the balanced sampler three equally-weighted sources.

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --output-dir /content/runs/stage2_w2a_balanced \
    --epochs 80 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 6e-4 --lr-backbone 1e-4 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 25 \
    --resume /content/runs/{W2_BASE}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w2a_balanced")

---
## W2B — Warmup heads from clean backbone (VDR recovery, Tier 1)

**Problem**: Stage 2A resumes from #25 which already has VDR=79% (down from 89.3% at stage1). Starting from the clean `stage1_conformer_128` backbone and using `--warmup-heads-epochs 40` lets pitch/VAD reconverge for 40 epochs before technique gradients compete.

**Expected**: VDR recovery toward 85%+, F1 comparable to W2A, better compound metric (F1 + VDR).

`--warmup-heads-epochs 40` = technique loss is zeroed for epochs 1–40, then normal joint training from epoch 41. The LR scheduler runs unaffected throughout.

In [ ]:
restore_from_drive(W1_BASE)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --warmup-heads-epochs 40 \
    --output-dir /content/runs/stage2_w2b_warmup \
    --epochs 120 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 6e-4 --lr-backbone 1e-4 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 30 \
    --resume /content/runs/{W1_BASE}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w2b_warmup")

---
## W2C — Dataset curriculum: GTSinger-only pretrain → combined fine-tune (Tier 2)

**Why**: Training combined from epoch 1 lets VocalSet artifacts (recording conditions, singer identity cues from 824 clips) dominate the backbone from the start. Pretraining on GTSinger only (10k+ clips, diverse singers, diverse recording conditions) for 50 epochs first gives the backbone a more general foundation before VocalSet data enters.

**How `--pretrain-data-dir` works**:
- Epochs 1–50: train only on `--pretrain-data-dir` + `--pretrain-technique-dirs` (GTSinger clean + GTSinger-tech)
- Epochs 51+: switch to `--data-dir` + `--technique-dirs` (combined with `--balance-datasets`)
- The LR schedule spans all epochs continuously — no restarts

**Expected**: best GTSinger generalization; GTSinger mF1 target 0.40–0.50. Likely best compound metric of all W2 variants if VDR holds.

In [ ]:
# W1_BASE already restored in W2B cell above; if running standalone, uncomment:
# restore_from_drive(W1_BASE)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --pretrain-data-dir /content/data \
    --pretrain-epochs 50 \
    --pretrain-technique-dirs /content/data/gtsinger_technique \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --output-dir /content/runs/stage2_w2c_curriculum \
    --epochs 150 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 6e-4 --lr-backbone 1e-4 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 30 \
    --resume /content/runs/{W1_BASE}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w2c_curriculum")

---
## W2D — SupCon with reduced weight (fixing VDR crash, Tier 2)

**Problem in #27**: `--w-contrastive-technique 0.5` with full GTSinger-tech data crashed VDR by −10.6 pp. The SupCon loss pulls backbone representations into technique-discriminative directions that compete with pitch/VAD.

**Fix**: reduce SupCon weight to 0.1 so it acts as a regulariser rather than the dominant gradient. Also add `--metric-vdr-weight 2.0` so the checkpoint metric penalises VDR regression more heavily (score = F1 + 2×VDR instead of F1 + 1×VDR).

Run off best W2A checkpoint (balanced datasets already incorporated).

In [ ]:
# Run after W2A completes — restore if needed
# restore_from_drive("stage2_w2a_balanced")

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --contrastive-technique --w-contrastive-technique 0.1 --contrastive-temp 0.07 \
    --output-dir /content/runs/stage2_w2d_supcon_light \
    --epochs 50 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 2e-4 --lr-backbone 2e-5 \
    --w-vad 0.05 --w-pitch 2 --w-technique 0.5 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 2.0 \
    --patience 20 \
    --resume /content/runs/stage2_w2a_balanced/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w2d_supcon_light")

---
## V4 — Note segmentation head (Variant 4, Tier 3)

Adds `head_note_onset` and `head_note_offset` — two binary frame-level classifiers.

**Prerequisites**: `data/annotated_vocalset/note_train.npz` must exist with keys:
- `mel`: (total_frames, 40) float16
- `onset`: (total_frames,) float32 — 1.0 at note onset frames
- `offset`: (total_frames,) float32 — 1.0 at note offset frames
- `lengths`: (n_clips,) int32

If this file has the onset/offset keys, the cell below will work. If not, you need to run `scripts/extractNotes.py` first to generate it from MIDI-aligned annotations.

Run off the best W2 checkpoint (`W2_BEST` — update after W2A/B/C results are known).

In [ ]:
import numpy as np

# Verify note_train.npz has the required onset/offset keys
note_path = '/content/data/annotated_vocalset/note_train.npz'
d = np.load(note_path)
print(f"Keys: {list(d.keys())}")
has_onset_offset = 'onset' in d and 'offset' in d
print(f"onset/offset keys present: {has_onset_offset}")
if not has_onset_offset:
    print("WARNING: note_train.npz does not have onset/offset keys.")
    print("Run scripts/extractNotes.py to generate them from MIDI annotations.")

In [ ]:
# Update W2_BEST to whichever W2 run had the best compound metric (F1 + VDR)
W2_BEST = "stage2_w2a_balanced"   # replace after W2A/B/C results
# restore_from_drive(W2_BEST)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
                     /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --balance-datasets \
    --note-head \
    --note-dirs /content/data/annotated_vocalset \
    --w-note 0.5 \
    --output-dir /content/runs/stage2_v4_note \
    --epochs 60 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 2e-4 --lr-backbone 2e-5 \
    --w-vad 0.05 --w-pitch 2 --w-technique 0.5 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 20 \
    --resume /content/runs/{W2_BEST}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_v4_note")

---
## W2 — Download all checkpoints for local evaluation

Update `RUN_NAMES` after each W2 run completes to include new runs.

In [ ]:
import os, zipfile
from google.colab import files

RUN_NAMES = [
    "stage2_w2a_balanced",
    "stage2_w2b_warmup",
    "stage2_w2c_curriculum",
    "stage2_w2d_supcon_light",
    "stage2_v4_note",
]

KEEP = {"best_loss.pth", "best_metric.pth"}

for run in RUN_NAMES:
    src = f'/content/runs/{run}'
    if os.path.isdir(src):
        save_to_drive(run)

zip_path = '/content/w2_checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for run in RUN_NAMES:
        ckpt_dir = f'/content/runs/{run}/checkpoints'
        if not os.path.isdir(ckpt_dir):
            print(f"  SKIP (not found): {run}")
            continue
        for fname in KEEP:
            fpath = os.path.join(ckpt_dir, fname)
            if not os.path.exists(fpath):
                print(f"  SKIP (missing): {run}/{fname}")
                continue
            arcname = f'vocalcoach/runs/{run}/checkpoints/{fname}'
            zf.write(fpath, arcname=arcname)
            print(f"  + {arcname}  ({os.path.getsize(fpath)/1e6:.1f} MB)")

print(f"\nZip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("Extract with:  unzip w2_checkpoints.zip  (from ~/NanoPitch-MusicalAI/)")
files.download(zip_path)